Precisamos conocer:
1) El porcentaje de personas que completó cada formulario (en este caso son solo 2 personas, pero el reporte debe considerar que en un futuro próximo pueden ser más y ese % siempre se debe calcular sobre el total de personas). Todos los usuarios deben completar todos los formularios. 

In [3]:
from pathlib import Path
import pandas as pd

# Raíz del proyecto
try:
    BASE_DIR = Path(__file__).parent
except NameError:
    BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"

# Carpeta de datos
DATA_DIR = BASE_DIR / "data"

# Buscar archivo Excel
excel_files = list(DATA_DIR.glob("*.xlsx"))

if not excel_files:
    raise FileNotFoundError("No se encontró ningún archivo .xlsx en la carpeta data")

# Tomamos el primero
excel_path = excel_files[0]

# Validar hojas existentes
xls = pd.ExcelFile(excel_path)
required_sheets = {
    "Formularios totales",
    "Formularios completados"
}

missing = required_sheets - set(xls.sheet_names)
if missing:
    raise ValueError(f"Faltan hojas requeridas: {missing}")

# Cargar hojas
df_formularios = pd.read_excel(
    excel_path,
    sheet_name="Formularios totales"
)

df_completados = pd.read_excel(
    excel_path,
    sheet_name="Formularios completados"
)


Calculos y display

In [24]:
# Total de formularios obligatorios
total_formularios = df_formularios["id"].nunique()

# Total de personas (se asume que todos deben completar)
total_usuarios = df_completados["userId"].nunique()

# Usuarios únicos que completaron cada tipo de formulario
usuarios_por_formulario = (
    df_completados
        .groupby("parentId")["userId"]
        .nunique()
        .reset_index(name="Completados")
)

# Unir con formularios totales por izquierda (para incluir los nunca completados)
df_final = df_formularios.merge(
    usuarios_por_formulario,
    left_on="id",
    right_on="parentId",
    how="left"
)

# Completar la cantidad de forms que no fueron completados con 0
df_final["Completados"] = df_final["Completados"].fillna(0).astype(int)

df_final["Faltantes"] = total_usuarios - df_final["Completados"]

df_final.rename(columns={"title": "Tipo de formulario", "id": "id_formulario"}, inplace=True)

# Calcular porcentaje (siempre sobre total de usuarios)
df_final["% completado"] = (
    df_final["Completados"] / total_usuarios * 100
).round(1).astype(str) + "%"

print(" Porcentaje de formularios completados por tipo de formulario")
display(df_final[["id_formulario", "Tipo de formulario", "Completados", "Faltantes", "% completado"]])

 Porcentaje de formularios completados por tipo de formulario


,id_formulario,Tipo de formulario,Completados,Faltantes,% completado
0,1465783,Declaración Jurada de Domicilio,1,1,50.0%
1,1465805,Código de ética,1,1,50.0%
2,1465792,Boletín Informativo Sistema de Pensiones,1,1,50.0%
3,1465784,Declaración Jurada Medidas Preventivas Covid 1...,0,2,0.0%
4,1465785,Política corporativa de prevención de corrupci...,1,1,50.0%
5,1465791,Reglamento interno de Trabajo - RIT Aprobado D...,1,1,50.0%
6,1465786,Política corporativa de conflicto de Intereses.,1,1,50.0%
7,1465815,Declaración Jurada Patrimonial,2,0,100.0%
8,1465787,Política de Compensaciones,1,1,50.0%
9,1465788,Declaración de consentimiento para tratamiento...,1,1,50.0%


2) El porcentaje de formularios que completó cada persona. Todos los usuarios deben completar todos los formularios. 

In [ ]:
# Agrupo pero esta vez por usuarios únicos
formularios_por_usuario = (
    df_completados
        .groupby("userId")["parentId"]
        .nunique()
        .reset_index(name="Formularios completados")
)

# Calculo el porcentaje sobre el total de formularios (ya calculado antes)
formularios_por_usuario["% completado"] = (
    formularios_por_usuario["Formularios completados"] / total_formularios * 100
).round(1).astype(str) + "%"

# Formateo final para el display
df_usuario_final = formularios_por_usuario.sort_values(
    "% completado",
    ascending=False)

df_usuario_final["% completado"] = (
    df_usuario_final["% completado"].astype(str) + "%")

df_usuario_final.rename(
    columns={"userId": "ID Usuario"},
    inplace=True)

df_usuario_final["Formularios faltantes"] = (total_formularios - formularios_por_usuario["Formularios completados"])

print(" Porcentaje de formularios completados por usuario")
display(df_usuario_final[["ID Usuario","Formularios faltantes", "Formularios completados", "% completado"]])

 Porcentaje de formularios completados por usuario


KeyError: "['userID'] not in index"

En un simple dashboard / reporte se debe mostrar qué formularios tiene pendiente cada usuario.	